[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [asyncpg and psycopg3, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/asyncpg-and-psycopg3-deep-dive.html)

# Which Driver &middot; Solutions


One way to do each task. Not the only way. If yours runs and does what was asked, yours is
right too.

The first cell is the notebook's Setup, with `SMALL`, `WIDE`, `SLOW` and the four ways of running
them. Run it first. Every task reports a band rather than a number, and they can be run in any
order.


In [1]:
import asyncio
import gc
import getpass
import os
import subprocess
import sys
import time
import warnings
from importlib.metadata import PackageNotFoundError, version

try:
    if version("psycopg") < "3.3" or version("asyncpg") < "0.31":
        raise PackageNotFoundError
except PackageNotFoundError:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--root-user-action=ignore",
                    "psycopg[binary,pool]==3.3.6", "psycopg-pool==3.3.2", "asyncpg==0.31.0"],
                   check=True)

import asyncpg
import psycopg
import psycopg_pool

def shell(command):
    """Run a shell command and hand back what it printed, without letting it stop the notebook."""
    done = subprocess.run(command, shell=True, capture_output=True, text=True)
    return done.returncode, (done.stdout + done.stderr).strip()


def answering(database="postgres"):
    """Whether a server is there, asked the only way that needs no client binaries."""
    try:
        with psycopg.connect(f"dbname={database}", connect_timeout=2):
            return True
    except psycopg.OperationalError:
        return False


def start_server(wait=60):
    """Install and start PostgreSQL if nothing is answering. Returns what it had to do."""
    if answering():
        return "already running"
    if sys.platform != "linux":
        raise RuntimeError("No PostgreSQL is answering. Start your own server and run this again: "
                           "this cell only installs one on Linux, which is what Colab runs.")

    sudo = "" if os.geteuid() == 0 else "sudo "
    shell(f"{sudo}apt-get -qq update")
    shell(f"{sudo}apt-get -qq -y install postgresql postgresql-contrib")
    shell(f"{sudo}service postgresql start")                        # Colab has no systemd

    for attempt in range(1, wait + 1):                              # start returns before it listens
        if shell("pg_isready -q")[0] == 0:
            break
        print(f"  waiting for the cluster ({attempt})")              # a silent minute looks hung
        time.sleep(1)
    else:
        raise RuntimeError(f"PostgreSQL did not accept connections within {wait} seconds.")

    me = getpass.getuser()                                          # peer authentication wants a role
    asking = f"""sudo -u postgres psql -tAc "SELECT 1 FROM pg_roles WHERE rolname='{me}'" """
    if shell(asking)[1] != "1":                                     # named for the operating system user
        shell(f"sudo -u postgres createuser -s {me}")
    return "installed and started"

def build(rows=5000):
    """Make the guide database and its events table, and fill it once."""
    with psycopg.connect("dbname=postgres", autocommit=True) as conn:
        if not conn.execute("SELECT 1 FROM pg_database WHERE datname = 'guide'").fetchone():
            conn.execute("CREATE DATABASE guide")                   # cannot run in a transaction

    with psycopg.connect("dbname=guide", autocommit=True) as conn:
        for (leftover,) in conn.execute(                            # whatever an earlier run made
                "SELECT tablename FROM pg_tables "
                "WHERE schemaname = 'public' AND tablename <> 'events'").fetchall():
            conn.execute(f'DROP TABLE IF EXISTS "{leftover}" CASCADE')

        conn.execute("""CREATE TABLE IF NOT EXISTS events (
                            id bigserial PRIMARY KEY,
                            ts timestamptz NOT NULL DEFAULT now(),
                            kind text NOT NULL,
                            payload jsonb NOT NULL)""")
        if conn.execute("SELECT count(*) FROM events").fetchone()[0] == 0:
            conn.execute("""INSERT INTO events (kind, payload)
                            SELECT (ARRAY['click', 'view', 'purchase'])[1 + n %% 3],
                                   jsonb_build_object('n', n, 'size', 1 + n %% 7)
                            FROM generate_series(1, %s) AS n""", (rows,))
        return conn.execute("SELECT count(*) FROM events").fetchone()[0]

def report():
    """One line naming what this notebook is running against."""
    rows = build()                                                  # makes the database if it is new
    with psycopg.connect("dbname=guide") as conn:
        major = int(conn.execute("SHOW server_version_num").fetchone()[0]) // 10000
    return (f"PostgreSQL {major} | psycopg {version('psycopg')} | asyncpg {version('asyncpg')} "
            f"| events: {rows} rows")

SMALL = "SELECT count(*) FROM events WHERE kind = 'click'"
WIDE = "SELECT id, ts, kind, payload FROM events"
SLOW = "SELECT pg_sleep(0.01), 1"                                   # ten milliseconds of waiting


def compared(baseline, measured):
    """A band, not a number: a ratio on a shared machine is not repeatable to two decimals."""
    ratio = baseline / measured
    if ratio < 1.2:
        return "about the same"
    return "somewhat faster" if ratio < 3 else "several times faster"


def best_of(run, repeats=3):
    """The fastest of several runs, which is the least noisy summary of a timing."""
    return min(run() for _ in range(repeats))


def blocking(sql, runs):
    """psycopg, one connection, one query at a time. This is the baseline everything is against."""
    def once():
        with psycopg.connect("dbname=guide", autocommit=True) as conn:
            start = time.perf_counter()
            for _ in range(runs):
                conn.execute(sql).fetchall()
            return time.perf_counter() - start
    return best_of(once)


async def best_of_async(run, repeats=3):
    return min([await run() for _ in range(repeats)])


async def asyncpg_serial(sql, runs):
    async def once():
        conn = await asyncpg.connect(database="guide")
        start = time.perf_counter()
        for _ in range(runs):
            await conn.fetch(sql)
        taken = time.perf_counter() - start
        await conn.close()
        return taken
    return await best_of_async(once)


async def asyncpg_pooled(sql, runs, size=8):
    async def once():
        pool = await asyncpg.create_pool(database="guide", min_size=size, max_size=size)
        start = time.perf_counter()
        await asyncio.gather(*(pool.fetch(sql) for _ in range(runs)))
        taken = time.perf_counter() - start
        await pool.close()
        return taken
    return await best_of_async(once)


async def psycopg_pooled(sql, runs, size=8):
    async def once():
        pool = psycopg_pool.AsyncConnectionPool("dbname=guide", min_size=size, max_size=size,
                                                open=False)
        await pool.open(wait=True, timeout=10)

        async def one():
            async with pool.connection() as conn:
                return await (await conn.execute(sql)).fetchall()

        start = time.perf_counter()
        await asyncio.gather(*(one() for _ in range(runs)))
        taken = time.perf_counter() - start
        await pool.close()
        return taken
    return await best_of_async(once)


print("server:", start_server())
print(report())
print("measuring against a local Unix socket, which is the least favorable place for a driver")


server: already running
PostgreSQL 16 | psycopg 3.3.6 | asyncpg 0.31.0 | events: 5000 rows
measuring against a local Unix socket, which is the least favorable place for a driver


**1.** One small query, both drivers, nothing overlapping.


In [2]:
baseline = blocking(SMALL, 500)
print("psycopg, synchronous:  the baseline")
print("asyncpg, one at a time:", compared(baseline, await asyncpg_serial(SMALL, 500)))


psycopg, synchronous:  the baseline
asyncpg, one at a time: about the same


Round trips, and nothing else. Neither driver can be faster at waiting for a socket, so on this
workload there is nothing to choose between them.


**2.** A wide result, both drivers, still nothing overlapping.


In [3]:
baseline = blocking(WIDE, 20)
print("psycopg, synchronous:  the baseline")
print("asyncpg, one at a time:", compared(baseline, await asyncpg_serial(WIDE, 20)))


psycopg, synchronous:  the baseline
asyncpg, one at a time: several times faster


The same two drivers on the same machine with no concurrency, and now a real difference. Decoding a
hundred thousand values is where asyncpg's own protocol implementation earns its place.


**3.** The two pools, against each other.


In [4]:
psycopg_time = await psycopg_pooled(SMALL, 500)
asyncpg_time = await asyncpg_pooled(SMALL, 500)

print("psycopg's async pool: the baseline for this comparison")
print("asyncpg's pool:      ", compared(psycopg_time, asyncpg_time))


psycopg's async pool: the baseline for this comparison
asyncpg's pool:       somewhat faster


Measuring the two pools against each other, rather than either against the synchronous baseline, is
what takes concurrency out of the answer. What is left is the drivers, and it is small.


**4.** With ten milliseconds of waiting in every query.


In [5]:
baseline = blocking(SLOW, 40)
print("psycopg, synchronous: the baseline (40 waits, one after another)")
print("psycopg, async pool: ", compared(baseline, await psycopg_pooled(SLOW, 40)))
print("asyncpg, async pool: ", compared(baseline, await asyncpg_pooled(SLOW, 40)))


psycopg, synchronous: the baseline (40 waits, one after another)
psycopg, async pool:  several times faster
asyncpg, async pool:  several times faster


The widest gap anywhere in this notebook, and the two drivers are level in it. This row is the
argument for a pool and for asynchronous code, and it is neutral on which driver provides them.


**5.** A pool of one.


In [6]:
baseline = blocking(SLOW, 40)
print("eight connections:", compared(baseline, await asyncpg_pooled(SLOW, 40, size=8)))
print("one connection:   ", compared(baseline, await asyncpg_pooled(SLOW, 40, size=1)))


eight connections: several times faster
one connection:    about the same


Forty queries and one connection is forty turns, whatever `asyncio.gather` was asked to do. The
concurrency was never in the driver: it was in the number of connections, which is what
**Connection Pools** was about.


**6.** How many connections each way opens.


In [7]:
def sessions():
    with psycopg.connect("dbname=guide", autocommit=True) as conn:
        return conn.execute("SELECT sessions FROM pg_stat_database "
                            "WHERE datname = 'guide'").fetchone()[0]


before = sessions()
blocking(SMALL, 10)
one_connection = sessions() - before - 1                            # less the one that asked

before = sessions()
await asyncpg_pooled(SMALL, 10, size=8)
a_pool = sessions() - before - 1

print("psycopg, synchronous, 10 queries:", one_connection, "connections")
print("asyncpg, pooled, 10 queries:     ", a_pool, "connections")
print("the pool opens more, once, and then stops")


psycopg, synchronous, 10 queries: 3 connections
asyncpg, pooled, 10 queries:      24 connections
the pool opens more, once, and then stops


The pooled run opens more connections and the serial run opens fewer, which is the trade concurrency
makes with the server. `best_of` runs each measurement three times, so the counts are three pools
rather than one, and that is worth noticing before reading any number here as a per-request cost.


---

&#8592; **Back to:** [Which Driver](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/asyncpg-and-psycopg3-deep-dive/16-which-driver.ipynb)  &nbsp;&middot;&nbsp;  [asyncpg and psycopg3, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/asyncpg-and-psycopg3-deep-dive.html)
